# Phase 6: Experiments and the Critical Adoption Share A*

Goal: sweep mu (price-impact coefficient), phi (residual AR coefficient), and the rolling forecast-window length w, and for each (w, phi, mu) combination locate the critical adoption share A* at which the proposal's mechanism has bitten. Produce a pair of three-panel heatmaps (one panel per w) and a small summary table. CLAUDE.md describes phase 6 as the compact headline result of the project.

Reference: section 4.2 of the proposal for A* and the threshold metrics, section 3.5 for stochastic diffusion, and phase 4 for the residual-vs-realised R^2 distinction.

**A note on the threshold.** The proposal defines A* as the smallest A at which the rolling out-of-sample R^2 crosses zero, or net profit crosses zero. In this cost-free baseline the residual R^2 falls but stays positive (typically ~0.025 even at full adoption) and adopter mean profit stays positive (it grows with adoption, by analytical inspection). So an absolute zero crossing does not fire. The cells below use **relative** thresholds instead: A*_R2 is the smallest A at which the rolling residual R^2 has fallen to half its baseline value, and A*_phi is the smallest A at which the rolling effective phi has grown to 1.5x its baseline. Once transaction costs land in phase 7 the absolute zero-crossing definition becomes operational and we swap back.

**Two correctness fences.** Crossings are searched only at indices past the baseline window (otherwise rolling-metric noise inside the low-adoption window can satisfy the relative threshold before adoption has done anything), and any cell whose hit rate (fraction of seeds in which a crossing actually happened) falls below 50% is reported as NaN to avoid a noisy minority of seeds setting the cell. Both raw matrices are saved alongside the filtered cells.

**Speed.** The seed loop is parallelised with `joblib.Parallel(n_jobs=-1)`. The simulator is pure and deterministic given seed, so it parallelises for free. On an 8-core machine the full 3 x 6 x 5 x 10 = 900-run sweep finishes in ~5 minutes.

In [ ]:
# Common run parameters.
N = 200
T = 8000
sigma_news = 0.01
sigma_q = 1.0
forecast_p = 1
risk_scale = 0.001
q_cap = 0.05
eval_window = 1000
adoption_pi = 3e-4   # slow stochastic diffusion: A ramps from 0 to ~0.87 over the run

# Sweep grid (lists; converted to numpy in the run cell).
mu_grid = [0.025, 0.05, 0.075, 0.10, 0.15]
phi_grid = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
w_grid = [100, 250, 500]

# Number of seeds per (w, phi, mu) cell. With joblib parallelism the
# 3 x 6 x 5 x 10 = 900 runs finish in ~5 min on 8 cores.
num_seeds_phase6 = 10
base_seed = 2000

# Relative thresholds for A*. The R^2 threshold is hit when the rolling
# residual R^2 has dropped to half its baseline; the phi threshold is
# hit when the rolling effective phi has grown by 50%.
r2_threshold_factor = 0.5
phi_threshold_factor = 1.5

# Cells with hit rate below this are reported as NaN: a noisy minority of
# seeds crossing the threshold should not pretend to set the cell.
min_hit_rate = 0.5

# Length of the low-adoption baseline window after the rolling-metric warmup.
baseline_span = 1500

fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from reflexive_market import simulate
from reflexive_market.metrics import rolling_oos_r2, rolling_phi

In [ ]:
def find_a_star(adoption_share, metric, threshold, direction, start_idx=0):
    """Smallest adoption share at indices >= start_idx where metric crosses threshold.

    direction == 'below': smallest A where metric < threshold (R^2 erosion).
    direction == 'above': smallest A where metric > threshold (phi growth).
    Restricting to indices past the baseline window keeps rolling-metric noise
    inside the low-adoption span from satisfying the threshold before adoption
    has had a chance to erode anything.
    """
    A = adoption_share[start_idx:]
    M = metric[start_idx:]
    if direction == "below":
        crossed = np.isfinite(M) & (M < threshold)
    else:
        crossed = np.isfinite(M) & (M > threshold)
    if not np.any(crossed):
        return np.nan
    return float(A[crossed].min())


def run_seed(seed_s, fw, adoption_start_t_w, baseline_lo_w, baseline_hi_w,
             mu_v, phi_v):
    """One simulator run + threshold detection for one seed and one cell."""
    rng_s = np.random.default_rng(seed_s)
    out = simulate.run(
        T=T, N=N, mu=mu_v, phi=phi_v,
        sigma_news=sigma_news, sigma_q=sigma_q,
        rng=rng_s,
        forecast_window=fw, forecast_p=forecast_p,
        risk_scale=risk_scale, q_cap=q_cap,
        adoption_pi=adoption_pi, adoption_delta=0.0,
        adoption_start_t=adoption_start_t_w,
    )
    A = out["adoption_share"]
    residual = out["returns"] - mu_v * out["demand"]
    R2 = rolling_oos_r2(residual, out["forecasts"], eval_window)
    phi_eff = rolling_phi(out["returns"], eval_window)

    base_R2 = float(np.nanmean(R2[baseline_lo_w:baseline_hi_w]))
    base_phi = float(np.nanmean(phi_eff[baseline_lo_w:baseline_hi_w]))

    a_star_R2_s = (
        find_a_star(A, R2, r2_threshold_factor * base_R2, "below",
                    start_idx=baseline_hi_w)
        if base_R2 > 0 else np.nan
    )
    a_star_phi_s = (
        find_a_star(A, phi_eff, phi_threshold_factor * base_phi, "above",
                    start_idx=baseline_hi_w)
        if base_phi > 0 else np.nan
    )
    return base_R2, base_phi, a_star_R2_s, a_star_phi_s

In [ ]:
mu_arr = np.array(mu_grid)
phi_arr = np.array(phi_grid)
w_arr = np.array(w_grid, dtype=int)
n_mu = mu_arr.size
n_phi = phi_arr.size
n_w = w_arr.size

# Build a flat job list of (k, i, j, seed_s, fw, adoption_start_t_w,
# baseline_lo_w, baseline_hi_w, mu_v, phi_v) tuples. Flat is the only
# shape that lets joblib fully saturate the workers.
jobs = []
for k, w_v in enumerate(w_arr):
    fw = int(w_v)
    adoption_start_t_w = fw + eval_window
    baseline_lo_w = adoption_start_t_w
    baseline_hi_w = adoption_start_t_w + baseline_span
    for i, phi_v in enumerate(phi_arr):
        for j, mu_v in enumerate(mu_arr):
            for s in range(num_seeds_phase6):
                seed_s = base_seed + 100000 * k + 1000 * (i * n_mu + j) + s
                jobs.append((k, i, j, seed_s, fw, adoption_start_t_w,
                              baseline_lo_w, baseline_hi_w,
                              float(mu_v), float(phi_v)))

print(f"Submitting {len(jobs)} simulator runs across {n_w}*{n_phi}*{n_mu} = "
      f"{n_w*n_phi*n_mu} cells, {num_seeds_phase6} seeds each")

raw = Parallel(n_jobs=-1, verbose=5)(
    delayed(run_seed)(seed_s, fw, adoption_start_t_w, baseline_lo_w,
                       baseline_hi_w, mu_v, phi_v)
    for (_, _, _, seed_s, fw, adoption_start_t_w, baseline_lo_w,
         baseline_hi_w, mu_v, phi_v) in jobs
)

In [ ]:
# Aggregate per (k, i, j) cell.
a_star_R2 = np.full((n_w, n_phi, n_mu), np.nan)
a_star_phi = np.full((n_w, n_phi, n_mu), np.nan)
hit_rate_R2 = np.zeros((n_w, n_phi, n_mu))
hit_rate_phi = np.zeros((n_w, n_phi, n_mu))
baseline_R2_grid = np.full((n_w, n_phi, n_mu), np.nan)
baseline_phi_grid = np.full((n_w, n_phi, n_mu), np.nan)

buckets = {}
for (k, i, j, *_), result in zip(jobs, raw):
    key = (k, i, j)
    buckets.setdefault(key, []).append(result)

for (k, i, j), seeds in buckets.items():
    base_R2s = np.array([r[0] for r in seeds])
    base_phis = np.array([r[1] for r in seeds])
    a_R2s = np.array([r[2] for r in seeds])
    a_phis = np.array([r[3] for r in seeds])

    baseline_R2_grid[k, i, j] = float(np.nanmean(base_R2s))
    baseline_phi_grid[k, i, j] = float(np.nanmean(base_phis))

    valid_R2 = base_R2s > 0
    valid_phi = base_phis > 0
    if valid_R2.any():
        crossed = valid_R2 & np.isfinite(a_R2s)
        hit_rate_R2[k, i, j] = crossed.sum() / valid_R2.sum()
        if hit_rate_R2[k, i, j] >= min_hit_rate:
            a_star_R2[k, i, j] = float(np.mean(a_R2s[crossed]))
    if valid_phi.any():
        crossed = valid_phi & np.isfinite(a_phis)
        hit_rate_phi[k, i, j] = crossed.sum() / valid_phi.sum()
        if hit_rate_phi[k, i, j] >= min_hit_rate:
            a_star_phi[k, i, j] = float(np.mean(a_phis[crossed]))

print(f"Sweep complete: {n_w}x{n_phi}x{n_mu} cells, {num_seeds_phase6} seeds each.")

In [ ]:
for k, w_v in enumerate(w_arr):
    print(f"\n--- w = {w_v} ---")
    print(f"{'phi':>6} | {'mu':>8} | {'A*_R2':>8} | {'hitR2':>6} | "
          f"{'A*_phi':>8} | {'hitphi':>7} | {'baseR2':>9} | {'basephi':>9}")
    print("-" * 84)
    for i, phi_v in enumerate(phi_arr):
        for j, mu_v in enumerate(mu_arr):
            print(
                f"{phi_v:>6.2f} | {mu_v:>8.3f} | "
                f"{a_star_R2[k, i, j]:>8.3f} | {hit_rate_R2[k, i, j]:>6.2f} | "
                f"{a_star_phi[k, i, j]:>8.3f} | {hit_rate_phi[k, i, j]:>7.2f} | "
                f"{baseline_R2_grid[k, i, j]:>9.4f} | {baseline_phi_grid[k, i, j]:>9.4f}"
            )

## Heatmap 1: A*_R2 across (mu, phi, w)

Smallest adoption share at which the rolling residual OOS R^2 has fallen to 50% of its low-adoption baseline. Three panels for three forecast-window lengths. Darker cells mean erosion bites at lower adoption (the mechanism is stronger). White / NaN cells mean the threshold is not crossed by enough seeds inside the observable adoption range; the inset shows the actual hit rate.

In [ ]:
fig, axes = plt.subplots(1, n_w, figsize=(6 * n_w, 5), sharey=True)
for k, w_v in enumerate(w_arr):
    ax = axes[k] if n_w > 1 else axes
    im = ax.imshow(a_star_R2[k], origin="lower", cmap="viridis_r",
                    vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_xticks(range(n_mu))
    ax.set_xticklabels([f"{m:g}" for m in mu_arr])
    if k == 0:
        ax.set_yticks(range(n_phi))
        ax.set_yticklabels([f"{p:.2f}" for p in phi_arr])
        ax.set_ylabel("phi (input AR coefficient)")
    ax.set_xlabel("mu (price-impact coefficient)")
    ax.set_title(f"w = {w_v}")
    for i in range(n_phi):
        for j in range(n_mu):
            v = a_star_R2[k, i, j]
            if not np.isfinite(v):
                hr = hit_rate_R2[k, i, j]
                text = f"hit {hr:.1f}"
                color = "black"
            else:
                text = f"{v:.2f}"
                color = "white" if v < 0.5 else "black"
            ax.text(j, i, text, ha="center", va="center", color=color, fontsize=8)
    fig.colorbar(im, ax=ax, label="A*")
fig.suptitle(
    f"A* at residual R^2 = {r2_threshold_factor} x baseline, hit rate >= {min_hit_rate} "
    f"(over {num_seeds_phase6} seeds)",
    y=1.02,
)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_a_star_R2_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Heatmap 2: A*_phi across (mu, phi, w)

Smallest adoption share at which the rolling effective phi has grown to 150% of its low-adoption baseline. Same mechanism viewed through autoregressive amplification rather than forecast erosion. Darker cells mean amplification bites earlier.

In [ ]:
fig, axes = plt.subplots(1, n_w, figsize=(6 * n_w, 5), sharey=True)
for k, w_v in enumerate(w_arr):
    ax = axes[k] if n_w > 1 else axes
    im = ax.imshow(a_star_phi[k], origin="lower", cmap="viridis_r",
                    vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_xticks(range(n_mu))
    ax.set_xticklabels([f"{m:g}" for m in mu_arr])
    if k == 0:
        ax.set_yticks(range(n_phi))
        ax.set_yticklabels([f"{p:.2f}" for p in phi_arr])
        ax.set_ylabel("phi (input AR coefficient)")
    ax.set_xlabel("mu (price-impact coefficient)")
    ax.set_title(f"w = {w_v}")
    for i in range(n_phi):
        for j in range(n_mu):
            v = a_star_phi[k, i, j]
            if not np.isfinite(v):
                hr = hit_rate_phi[k, i, j]
                text = f"hit {hr:.1f}"
                color = "black"
            else:
                text = f"{v:.2f}"
                color = "white" if v < 0.5 else "black"
            ax.text(j, i, text, ha="center", va="center", color=color, fontsize=8)
    fig.colorbar(im, ax=ax, label="A*")
fig.suptitle(
    f"A* at effective phi = {phi_threshold_factor} x baseline, hit rate >= {min_hit_rate} "
    f"(over {num_seeds_phase6} seeds)",
    y=1.02,
)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_a_star_phi_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Save numeric summary

3D arrays for both A* metrics (filtered by hit rate), the raw hit rate matrices, the parameter grids, and the run inputs. Small npz, fully self-contained for replotting.

In [ ]:
np.savez(
    f"{data_dir}/phase_06_a_star_grid.npz",
    mu_grid=mu_arr,
    phi_grid=phi_arr,
    w_grid=w_arr,
    a_star_R2=a_star_R2,
    a_star_phi=a_star_phi,
    hit_rate_R2=hit_rate_R2,
    hit_rate_phi=hit_rate_phi,
    baseline_R2_grid=baseline_R2_grid,
    baseline_phi_grid=baseline_phi_grid,
    eval_window=np.array(eval_window),
    risk_scale=np.array(risk_scale),
    q_cap=np.array(q_cap),
    adoption_pi=np.array(adoption_pi),
    r2_threshold_factor=np.array(r2_threshold_factor),
    phi_threshold_factor=np.array(phi_threshold_factor),
    min_hit_rate=np.array(min_hit_rate),
    baseline_span=np.array(baseline_span),
    T=np.array(T),
    N=np.array(N),
    num_seeds=np.array(num_seeds_phase6),
    base_seed=np.array(base_seed),
)

## Done when

- The two heatmaps populate the bulk of the (mu, phi) grid for each w; cells that fail the hit-rate filter print `hit X.X` instead of a number, so an honest reader can tell empty cells from low-A* cells.
- The heatmaps tell a coherent monotonic story per panel: larger mu and larger phi push A* down (erosion bites earlier).
- The two heatmaps are visually consistent (same cells dark in both): residual-R^2 erosion and effective-phi amplification are two views of the same mechanism.
- Across panels (w varying) the pattern is qualitatively similar: w mainly shifts when the rolling fit catches up to phi_eff, which moves A* but does not flip the sign of the gradient.

Phase 7 will add transaction costs and the absolute *net profit crosses zero* version of A* becomes operational; rerun the same sweep with the cost term and a third heatmap appears that the cost-free baseline can't produce.